# 🌬️ AirSense AI — Air Pollution Forecasting & Hotspot Intelligence
This notebook demonstrates the end-to-end Machine Learning pipeline for **1-Hour Ahead PM2.5 Forecasting**, risk classification, and actionable mitigation modeling using the **UCI Beijing Multi-Site Air Quality Dataset**.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import joblib

# Paths
BASE_DIR = os.path.abspath('..')
DATA_DIR = os.path.join(BASE_DIR, 'data', 'processed')
MODELS_DIR = os.path.join(BASE_DIR, 'models')
print(f"Base directory: {BASE_DIR}")

## 1. Load Processed Datasets (Chronological Splits)
To prevent data leakage, training is strictly chronological:
- **Train (2013-2015)**: 298,080 rows
- **Validation (2016)**: 105,408 rows
- **Test (2017)**: 16,980 rows

In [ ]:
train_df = pd.read_parquet(os.path.join(DATA_DIR, 'train.parquet'))
val_df = pd.read_parquet(os.path.join(DATA_DIR, 'val.parquet'))
test_df = pd.read_parquet(os.path.join(DATA_DIR, 'test.parquet'))

print(f"Train shape: {train_df.shape}")
print(f"Val shape:   {val_df.shape}")
print(f"Test shape:  {test_df.shape}")

## 2. Model Performance & Evaluation Benchmark
Comparison of **Linear Regression (Baseline)**, **Random Forest**, and **XGBoost (Hist Tree)** on the unseen 2017 Test Set.

In [ ]:
with open(os.path.join(MODELS_DIR, 'model_metrics.json'), 'r') as f:
    metrics = json.load(f)

metrics_df = pd.DataFrame(metrics['metrics']).T
display(metrics_df)

## 3. Top Influential Features in Short-Term Forecasting

In [ ]:
with open(os.path.join(MODELS_DIR, 'evaluation_report.json'), 'r') as f:
    eval_rep = json.load(f)

print("Top Influential Features:")
for feat, score in eval_rep['top_features'].items():
    print(f"  - {feat:<24} : {score}")

## 4. Test Standalone Inference (Zero Flutter Required)

In [ ]:
import sys
sys.path.append(BASE_DIR)
from src.predict import AirSensePredictor

predictor = AirSensePredictor(models_dir=MODELS_DIR)
sample_telemetry = {
    "deviceId": "ESP32_AIR_01",
    "location": "Junction Central Corridor",
    "pm25": 86.4,
    "pm10": 124.2,
    "temperature": 28.5,
    "humidity": 62.0,
    "pressure": 1008.0,
    "wind_speed": 1.5,
    "co": 1200.0,
    "hour": 18
}

prediction = predictor.predict(sample_telemetry)
print(json.dumps(prediction, indent=2))